# Multi-Seed Validation — Reward-Conditioned Neural Memory Layer

**Purpose:** Validate the 71.4% improvement-on-failed result across 3 seeds.
**Runtime:** GPU (T4 16 GB), ~3 hours total (3 seeds × 5 sessions × 16 problems).
**Success criteria:** All 3 seeds show >40% improvement on failed problems.

| Seed | Expected outcome |
|------|-----------------|
| 42   | Baseline seed   |
| 123  | Variance check  |
| 456  | Variance check  |


In [1]:
import subprocess, sys
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "accelerate", "datasets", "faiss-cpu"],
    check=True
)

import os, json, torch, random, numpy as np
from pathlib import Path

sys.path.insert(0, "/kaggle/working/src")

for d in ["src", "problems", "memory_states", "results", "logs"]:
    Path(f"/kaggle/working/{d}").mkdir(parents=True, exist_ok=True)

print("Env ready. CUDA:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 88.7 MB/s eta 0:00:00
Env ready. CUDA: True
Device: Tesla T4


In [2]:
import shutil

SOURCE = "/kaggle/input/datasets/atromixtheanalyzer/memory-agent/src"
DEST   = "/kaggle/working/src"

if os.path.exists(SOURCE):
    for f in os.listdir(SOURCE):
        shutil.copy(os.path.join(SOURCE, f), os.path.join(DEST, f))
    print("Source files copied:", os.listdir(DEST))
else:
    print(f"WARNING: {SOURCE} not found.")


Source files copied: ['session_manager.py', 'model_surgery.py', 'sandbox.py', 'memory_module.py', 'eval.py']


In [3]:
from memory_module import run_all_isolation_tests
run_all_isolation_tests()


Running isolation tests on NeuralMemoryModule...
(All must pass before attaching to any LLM)

  PASS associative_storage: retrieval_error=0.0854
  PASS surprise_decay: 2.2200 -> 1.2913
  PASS reward_learning: all 3 pairs within 0.15 tolerance
  PASS generalisation: base=0.94, similar=0.92, diff=0.12
  PASS weight_norm_stability: 12.326 -> 7.879 (no NaN/Inf, retained >64% of initial norm)
         post-stress retrieval error: 0.1324 (functional: YES)

All isolation tests passed. Safe to proceed to LLM integration.


In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import login

import os
HF_TOKEN = os.environ.get("HF_TOKEN", "YOUR_HF_TOKEN")
MODEL_ID = "google/gemma-2-2b-it"

login(token=HF_TOKEN, add_to_git_credential=False)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)
print("Model loaded. Device:", next(model.parameters()).device)


config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

Model loaded. Device: cuda:0


In [5]:
from model_surgery import inject_memory_layers, measure_perplexity

VALIDATION_TEXTS = [
    "def add(a, b):\n    return a + b",
    "def reverse(s):\n    return s[::-1]",
    "for i in range(10):\n    print(i)",
    "x = [1, 2, 3]\nx.sort()\nprint(x)",
    "def is_prime(n):\n    return n > 1 and all(n % i for i in range(2, n))",
]

ppl_before = measure_perplexity(model, tokenizer, VALIDATION_TEXTS)
print(f"Perplexity before injection: {ppl_before:.4f}")

MEMORY_LAYERS = [4, 8, 16, 24]
model, memory_modules = inject_memory_layers(model, MEMORY_LAYERS)

ppl_after = measure_perplexity(model, tokenizer, VALIDATION_TEXTS)
delta_pct = abs(ppl_after - ppl_before) / ppl_before * 100
print(f"Perplexity after injection:  {ppl_after:.4f}")
print(f"Delta: {delta_pct:.2f}%  (must be < 1% to proceed)")

assert delta_pct < 1.0, f"FAIL: perplexity degraded by {delta_pct:.2f}%."
print("\nGate OK. Safe to proceed.")


Perplexity before injection: 1.7878
  Memory injected at layer 4 (device=cuda:0, dtype=torch.float16)
  Memory injected at layer 8 (device=cuda:0, dtype=torch.float16)
  Memory injected at layer 16 (device=cuda:1, dtype=torch.float16)
  Memory injected at layer 24 (device=cuda:1, dtype=torch.float16)
  Trainable: 191,158,280 / 2,805,500,168 (6.814%)
Perplexity after injection:  1.7894
Delta: 0.09%  (must be < 1% to proceed)

Gate OK. Safe to proceed.


In [6]:
from eval import load_humaneval, calibrate_problem_difficulty

PROBLEMS_PATH = "/kaggle/working/problems/calibrated_20.json"

if os.path.exists(PROBLEMS_PATH):
    with open(PROBLEMS_PATH) as f:
        calibrated = json.load(f)
    print(f"Loaded {len(calibrated)} calibrated problems from cache.")
else:
    raw_problems = load_humaneval("/kaggle/working/problems/humaneval_raw.json")
    print(f"Loaded {len(raw_problems)} raw HumanEval problems.")
    calibrated = calibrate_problem_difficulty(
        model, tokenizer,
        problems=raw_problems[:100],
        n_attempts=3,
        target_range=(0.3, 0.7),
        save_path=PROBLEMS_PATH
    )
    print(f"\nCalibrated set: {len(calibrated)} problems saved.")

print(f"\nUsing {len(calibrated)} problems for multi-seed evaluation.")


Local file not found — downloading from HuggingFace datasets...


README.md: 0.00B [00:00, ?B/s]

openai_humaneval/test-00000-of-00001.par(…):   0%|          | 0.00/83.9k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/164 [00:00<?, ? examples/s]

Saved 164 problems → /kaggle/working/problems/humaneval_raw.json
Loaded 164 raw HumanEval problems.
Calibrating 100 problems (target pass rate: 0.3–0.7)...

  DROP HumanEval/0: too easy  (pass_rate=1.00)
  DROP HumanEval/1: too easy  (pass_rate=1.00)
  KEEP HumanEval/2: pass_rate=0.33
  KEEP HumanEval/3: pass_rate=0.67
  DROP HumanEval/4: too easy  (pass_rate=1.00)
  KEEP HumanEval/5: pass_rate=0.33
  DROP HumanEval/6: too hard  (pass_rate=0.00)
  DROP HumanEval/7: too hard  (pass_rate=0.00)
  KEEP HumanEval/8: pass_rate=0.67
  DROP HumanEval/9: too easy  (pass_rate=1.00)
  DROP HumanEval/10: too hard  (pass_rate=0.00)
  DROP HumanEval/11: too easy  (pass_rate=1.00)
  DROP HumanEval/12: too easy  (pass_rate=1.00)
  KEEP HumanEval/13: pass_rate=0.67
  KEEP HumanEval/14: pass_rate=0.33
  DROP HumanEval/15: too easy  (pass_rate=1.00)
  DROP HumanEval/16: too easy  (pass_rate=1.00)
  DROP HumanEval/17: too hard  (pass_rate=0.00)
  KEEP HumanEval/18: pass_rate=0.67
  DROP HumanEval/19: too 

In [7]:
from session_manager import SessionManager
from eval import compute_improvement_rate
from memory_module import NeuralMemoryModule

SEEDS = [42, 123, 456]
N_SESSIONS = 5
all_seed_results = {}

# Save the initial (untrained) memory state so we can reset between seeds
initial_mem_states = {}
for idx, mem in memory_modules.items():
    initial_mem_states[idx] = {
        'layers': {k: v.clone() for k, v in mem.layers.state_dict().items()},
        'W_sa': {k: v.clone() for k, v in mem.W_sa.state_dict().items()},
        'momentum': {k: v.clone() for k, v in mem.momentum.items()},
    }

for seed in SEEDS:
    print(f"\n{'#'*60}")
    print(f"  SEED {seed}")
    print(f"{'#'*60}")

    # Set all random seeds
    random.seed(seed)
    np.random.seed(seed)

    # Reset memory modules to initial (untrained) state
    for idx, mem in memory_modules.items():
        mem.layers.load_state_dict(
            {k: v.clone() for k, v in initial_mem_states[idx]['layers'].items()}
        )
        mem.W_sa.load_state_dict(
            {k: v.clone() for k, v in initial_mem_states[idx]['W_sa'].items()}
        )
        mem.momentum = {
            k: v.clone() for k, v in initial_mem_states[idx]['momentum'].items()
        }

    # Clear any prior checkpoints for this seed
    user_id = f"seed_{seed}"
    mem_dir = Path(f"./memory_states/{user_id}")
    if mem_dir.exists():
        for f in mem_dir.glob("*.pt"):
            f.unlink()
        for f in mem_dir.glob("*.json"):
            f.unlink()

    seed_results = []
    for s in range(1, N_SESSIONS + 1):
        print(f"\n{'='*50}")
        print(f"SEED {seed} — SESSION {s}/{N_SESSIONS}")
        print(f"{'='*50}")

        mgr = SessionManager(model, tokenizer, memory_modules, user_id=user_id, temperature=0.2)
        summary = mgr.run_session(calibrated)
        summary["session"] = s
        summary["seed"] = seed
        seed_results.append(summary)
        mgr.log_diagnostics()

    all_seed_results[seed] = seed_results

    # Compute improvement for this seed
    impr, n_failed = compute_improvement_rate(
        seed_results[0]["results"],
        seed_results[-1]["results"]
    )
    print(f"\n  SEED {seed} SUMMARY: {impr*100:.1f}% improvement on {n_failed} failed problems")
    print(f"  Pass rate: {seed_results[0]['pass_rate']*100:.1f}% → {seed_results[-1]['pass_rate']*100:.1f}%")



############################################################
  SEED 42
############################################################

SEED 42 — SESSION 1/5
No prior memory state. Starting fresh.
  [FAIL] HumanEval/2: reward=0.00
  [PASS] HumanEval/3: reward=1.00
  [FAIL] HumanEval/5: reward=0.00
  [PASS] HumanEval/8: reward=1.00
  [PASS] HumanEval/13: reward=1.00
  [FAIL] HumanEval/14: reward=0.00
  [PASS] HumanEval/18: reward=1.00
  [PASS] HumanEval/24: reward=1.00
  [PASS] HumanEval/29: reward=1.00
  [FAIL] HumanEval/43: reward=0.67
  [FAIL] HumanEval/49: reward=0.14
  [PASS] HumanEval/54: reward=1.00
  [PASS] HumanEval/66: reward=1.00
  [FAIL] HumanEval/73: reward=0.88
  [PASS] HumanEval/89: reward=1.00
  [FAIL] HumanEval/93: reward=0.00
Memory saved: memory_20260527_111051.pt

Session complete: 56.2% (9/16)
Log: session_20260527_111052.json

── Memory Diagnostics ──
  Layer 4: weight_norm=2.1048
  Layer 8: weight_norm=1.4896
  Layer 16: weight_norm=0.3170
  Layer 24: weight_norm=9.

In [8]:
print("\n" + "="*70)
print("  MULTI-SEED AGGREGATE RESULTS")
print("="*70)

improvements = []
pass_trajectories = {}

for seed, results in all_seed_results.items():
    impr, n_failed = compute_improvement_rate(
        results[0]["results"],
        results[-1]["results"]
    )
    improvements.append(impr * 100)

    trajectory = [r["pass_rate"] * 100 for r in results]
    pass_trajectories[seed] = trajectory

    print(f"\nSeed {seed}:")
    print(f"  Sessions: {' → '.join(f'{t:.1f}%' for t in trajectory)}")
    print(f"  Improvement on failed: {impr*100:.1f}% (n={n_failed})")

mean_impr = np.mean(improvements)
std_impr  = np.std(improvements)

print(f"\n{'─'*40}")
print(f"  Mean improvement: {mean_impr:.1f}% ± {std_impr:.1f}%")
print(f"  Individual:       {', '.join(f'{x:.1f}%' for x in improvements)}")
print(f"  Target:           >40% (minimum), >60% (strong)")
print(f"{'─'*40}")

if mean_impr > 40:
    print("\n  ✅ RESULT IS STATISTICALLY DEFENSIBLE")
    print("     Ready for baselines and paper results section.")
else:
    print("\n  ⚠️ RESULT IS BELOW TARGET")
    print("     Investigate variance before running baselines.")

# Per-session mean across seeds
print("\n  Per-session mean pass rate:")
for s in range(N_SESSIONS):
    rates = [pass_trajectories[seed][s] for seed in SEEDS]
    print(f"    S{s+1}: {np.mean(rates):.1f}% ± {np.std(rates):.1f}%")



  MULTI-SEED AGGREGATE RESULTS

Seed 42:
  Sessions: 56.2% → 43.8% → 68.8% → 56.2% → 62.5%
  Improvement on failed: 57.1% (n=7)

Seed 123:
  Sessions: 62.5% → 62.5% → 43.8% → 62.5% → 50.0%
  Improvement on failed: 50.0% (n=6)

Seed 456:
  Sessions: 37.5% → 50.0% → 56.2% → 56.2% → 56.2%
  Improvement on failed: 40.0% (n=10)

────────────────────────────────────────
  Mean improvement: 49.0% ± 7.0%
  Individual:       57.1%, 50.0%, 40.0%
  Target:           >40% (minimum), >60% (strong)
────────────────────────────────────────

  ✅ RESULT IS STATISTICALLY DEFENSIBLE
     Ready for baselines and paper results section.

  Per-session mean pass rate:
    S1: 52.1% ± 10.6%
    S2: 52.1% ± 7.8%
    S3: 56.2% ± 10.2%
    S4: 58.3% ± 2.9%
    S5: 56.2% ± 5.1%


In [9]:
results_dir = Path("/kaggle/working/results")
results_dir.mkdir(parents=True, exist_ok=True)

# Save per-seed results
for seed, results in all_seed_results.items():
    path = results_dir / f"eval_seed_{seed}_results.json"
    with open(path, "w") as f:
        json.dump(results, f, indent=2, default=str)
    print(f"Saved: {path.name}")

# Save aggregate summary
summary = {
    "seeds": SEEDS,
    "n_sessions": N_SESSIONS,
    "n_problems": len(calibrated),
    "improvements": {str(s): imp for s, imp in zip(SEEDS, improvements)},
    "mean_improvement": round(mean_impr, 2),
    "std_improvement": round(std_impr, 2),
    "pass_trajectories": {str(s): t for s, t in pass_trajectories.items()},
}
with open(results_dir / "multiseed_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print("Saved: multiseed_summary.json")


Saved: eval_seed_42_results.json
Saved: eval_seed_123_results.json
Saved: eval_seed_456_results.json
Saved: multiseed_summary.json


In [10]:
import zipfile

ZIP_PATH = "/kaggle/working/multiseed_checkpoint.zip"

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for folder in ["memory_states", "results", "problems"]:
        root = f"/kaggle/working/{folder}"
        for dirpath, _, files in os.walk(root):
            for file in files:
                fpath = os.path.join(dirpath, file)
                zf.write(fpath, os.path.relpath(fpath, "/kaggle/working"))

size_mb = os.path.getsize(ZIP_PATH) / 1e6
print(f"Zipped → {ZIP_PATH}  ({size_mb:.1f} MB)")
print("Download via: Output panel → multiseed_checkpoint.zip → Download")


Zipped → /kaggle/working/multiseed_checkpoint.zip  (1641.7 MB)
Download via: Output panel → multiseed_checkpoint.zip → Download
